# Smoke tests for `geolab-gpu2`

For every package in `environment.yml` (conda + pip): try to import it and
exercise one minimal API call. CLI-only packages get a `which`/`--version`
check instead. A failure here means something installed but doesn't load,
which is usually a sign of an ABI mismatch or a missing system library.

Run all cells. The summary at the bottom lists pass/fail per package.

## Setup

In [ ]:
import sys

import test_helpers as test
from test_helpers import RESULTS, cli, py

test.RESULTS.clear()

print(f'Python {sys.version}')
print(f'sys.prefix: {sys.prefix}')

## Cloud & storage

In [ ]:
cli('aws')
py('awswrangler')
py('boto3', smoke=lambda m: m.client('s3', region_name='us-east-1'))
py('fsspec', smoke=lambda m: m.filesystem('memory'))
py('obstore')
py('s3fs', smoke=lambda m: m.S3FileSystem)

## Geospatial

In [ ]:
py('cartopy.crs', alias='cartopy',
   smoke=lambda m: m.PlateCarree())
py('contextily')
py('fiona', smoke=lambda m: m.supported_drivers)
py('folium',
   smoke=lambda m: m.Map(location=[0, 0], zoom_start=2))
py('osgeo.gdal', alias='gdal',
   smoke=lambda m: m.VersionInfo('RELEASE_NAME'))
py('ipyleaflet', smoke=lambda m: m.Map())
py('lonboard')
py('pyproj', smoke=lambda m: m.CRS('EPSG:4326'))
py('shapely.geometry', alias='shapely',
   smoke=lambda m: m.Point(0, 0).buffer(1).area)

## Core scientific stack

In [ ]:
py('numpy', smoke=lambda m: int(m.array([1, 2, 3]).sum()))
py('numba', smoke=lambda m: m.njit(lambda x: x + 1)(1))
py('scipy.stats', alias='scipy', smoke=lambda m: m.norm.cdf(0))
py('pandas',
   smoke=lambda m: m.DataFrame({'a': [1, 2]}).shape)
py('geopandas')
import matplotlib; matplotlib.use('Agg')
py('matplotlib', alias='matplotlib-base',
   smoke=lambda m: m.figure.Figure())
cli('dot', version_flag='-V')
py('pygraphviz', smoke=lambda m: m.AGraph().add_node(1))
py('xarray',
   smoke=lambda m: m.DataArray([1, 2, 3]).sum().item())
py('netCDF4', alias='netcdf4')
py('h5py')
py('h5netcdf')
py('pyarrow',
   smoke=lambda m: m.array([1, 2, 3]).to_pylist())
py('zarr',
   smoke=lambda m: m.zeros((3,), chunks=3, dtype='f4'))
py('virtualizarr')
py('bottleneck',
   smoke=lambda m: m.nansum([1.0, 2.0, float('nan'), 3.0]))
py('flox')
py('pooch')
py('dask.array', alias='dask',
   smoke=lambda m: m.ones(10, chunks=5).sum().compute())
py('distributed')
py('dask_gateway', alias='dask-gateway')
py('cvxpy', smoke=lambda m: m.Variable(name='x'))

## Geo / geoscience

In [ ]:
py('dascore')
cli('gmt', version_flag='--version')
py('obspy',
   smoke=lambda m: m.UTCDateTime('2020-01-01').timestamp)
py('obsplus')
py('pygmt')

## GPU / machine learning

In [ ]:
# PyTorch & seismic ML -----------------------------------------
# Informational only, not a pass/fail check: the driver is baked
# into this image, but the GPU device nodes still have to be
# passed through at `docker run` (--gpus all) or by the scheduler,
# so CUDA can legitimately be unavailable even on a correct build.
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  device: {torch.cuda.get_device_name(0)}')

py('torch',
   smoke=lambda m: m.tensor([1., 2., 3.]).sum().item())
py('seisbench', smoke=lambda m: m.models.PhaseNet())

# General ML / GPU tooling --------------------------------------
py('sklearn', alias='scikit-learn',
   smoke=lambda m: m.linear_model.LinearRegression())
py('skimage', alias='scikit-image',
   smoke=lambda m: m.data.camera())
py('xgboost',
   smoke=lambda m: m.XGBRegressor())
py('dask_ml', alias='dask-ml')
py('seaborn')
py('jupyterlab_nvdashboard', alias='jupyterlab-nvdashboard',
   smoke=lambda m: m._jupyter_labextension_paths())

# Profiling ------------------------------------------------------
py('line_profiler')
py('memory_profiler')
py('snakeviz')
py('watermark')

### GPU compute test

The checks above only prove the packages *import*, they pass on CPU-only
hosts too. This cell actually runs a computation on the GPU and fails loudly
if there isn't one to run it on, unlike the informational check above, this
one is a real pass/fail test.

Run this after `docker run --gpus all ...` (needs `nvidia-container-toolkit`
on the host), or in a GeoLab session that was actually scheduled onto a GPU
node.

In [ ]:
assert torch.cuda.is_available(), (
    'No GPU visible to this container. Re-run with `docker run --gpus all ...` '
    '(needs nvidia-container-toolkit on the host), or confirm a GPU was '
    'actually scheduled onto this session.'
)

device = torch.device('cuda')
a = torch.randn(4096, 4096, device=device)
b = torch.randn(4096, 4096, device=device)
c = a @ b
torch.cuda.synchronize()  # the matmul above is async; wait for it to finish

assert c.is_cuda
print(f'Ran a 4096x4096 matmul on: {torch.cuda.get_device_name(0)}')
print(f'Result: device={c.device}, shape={tuple(c.shape)}, c[0, 0]={c[0, 0].item():.4f}')
print('GPU compute test passed.')

## Phase association & location

Tools for grouping picks (from seisbench or any other picker) into events
and locating them, none of these run the actual picking above. `pyocto`,
`gamma` (GaMMA), and `pyrocko` are pip packages; `REAL`, `NLLoc`
(NonLinLoc), and `hypoDD` are legacy Fortran/C research codes with no
packaged installer, built from source in the Dockerfile. None of the
compiled tools have a `--version` flag -- `cli()` still passes as long as
the binary is on `$PATH` and responds at all, which a bare usage message
satisfies.

In [ ]:
# Association --------------------------------------------------
py('pyocto',
   smoke=lambda m: m.VelocityModel0D(p_velocity=6.0, s_velocity=3.5, tolerance=2.0))
py('gamma', alias='gmma',
   smoke=lambda m: m.GaussianMixture(n_components=1,
                                      station_locs=[[0.0, 0.0, 0.0], [1.0, 1.0, 0.0]],
                                      phase_type=['p', 's']))

# Waveform / location toolkit ------------------------------------
py('pyrocko')

# Location (built from source, see Dockerfile) --------------------
cli('REAL')
cli('pick2real')
cli('velest')
cli('NLLoc')
cli('Vel2Grid')
cli('Grid2Time')
cli('hypoDD')
cli('ph2dt')

## Utilities

In [ ]:
py('tqdm',
   smoke=lambda m: list(m.tqdm(range(3), disable=True)))
py('requests')
py('yaml', alias='pyyaml',
   smoke=lambda m: m.safe_load('a: 1'))
cli('gs', version_flag='--version')      # ghostscript
cli('ffmpeg', version_flag='-version')

## Dev tools

In [ ]:
cli('gh')
cli('gh-scoped-creds')
py('pytest')
cli('ruff')

## Jupyter stack & extensions

In [ ]:
py('jupyterhub')
py('jupyter_server')
py('jupyterlab')
py('ipykernel')
py('jupyter_resource_usage', alias='jupyter-resource-usage')
py('jupyter_ruff', alias='jupyter-ruff')
py('jupyter_server_proxy', alias='jupyter-server-proxy')
py('jupyterlab_git', alias='jupyterlab-git')
py('jupyterlab_myst', alias='jupyterlab-myst')
py('jupyterlab_code_formatter')
py('jupyterlab_pygments')
py('nbdime')

## pip packages & visualization

In [ ]:
# EarthScope --------------------------------------------------
py('earthscope_sdk', alias='earthscope-sdk')
cli('es')                       # earthscope-cli entry point
py('earthscopestraintools')

# Jupyter add-ons ---------------------------------------------
py('jupyterlab_jupyterbook_navigation')
py('jupyterlab_topbar_stop_server', alias='jupyterlab-topbar-stop-server',
   smoke=lambda m: m._jupyter_labextension_paths())

# Visualization & data frames ---------------------------------
py('altair',
   smoke=lambda m: m.Chart())
py('plotly')
py('polars',
   smoke=lambda m: m.DataFrame({'a': [1, 2]}))
py('vegafusion')
py('vl_convert', alias='vl-convert-python')
py('ipympl')
py('hvplot')
py('holoviews', alias='holoviews',
   smoke=lambda m: m.Curve([1, 2, 3]))
py('panel')

## Interactive widgets

In [ ]:
py('ipywidgets',
   smoke=lambda m: m.IntSlider(value=5, min=0, max=10))
py('anywidget')
py('bqplot')
py('ipytree', smoke=lambda m: m.Node(name='root'))
py('ipycytoscape', smoke=lambda m: m.CytoscapeWidget())
py('itables')
py('ipydatagrid')
from sidecar import Sidecar  # noqa: F401
py('sidecar')

## Summary

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(RESULTS,
                  columns=['package', 'status', 'version', 'error'])

passed = int((df['status'] == 'OK').sum())
total = len(df)
failed = total - passed

print(f'Results: {passed}/{total} OK, {failed} failed')
if failed:
    print('\nFailures:')
    for _, row in df[df['status'] == 'FAIL'].iterrows():
        print(f"  {row['package']:35s}  {row['error']}")

df.style.map(
    lambda v: ('color: red; font-weight: bold' if v == 'FAIL'
               else 'color: green'),
    subset=['status']
)